### SQL Database Processing ###

In [17]:

import sqlite3
import os

os.makedirs("data/databases", exist_ok=True)

In [18]:
## Create a SQLite database ##

conn = sqlite3.connect("data/databases/company.db")
cursor = conn.cursor()

In [19]:

cursor.execute("DROP TABLE IF EXISTS projects")
cursor.execute("DROP TABLE IF EXISTS employees")

cursor.execute('''
CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    role TEXT,
    department TEXT,
    salary REAL
)
''')

cursor.execute('''
CREATE TABLE projects (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    status TEXT,
    budget REAL,
    lead_id INTEGER
)
''')

conn.commit()

OperationalError: database is locked

### Add Some Data to table ###

In [ ]:
cursor.executemany('''
INSERT INTO employees (id, name, role, department, salary)
VALUES (?, ?, ?, ?, ?)
''', [
    (1, 'Arun Kumar', 'Software Engineer', 'IT', 60000),
    (2, 'Priya Sharma', 'Tech Lead', 'IT', 90000),
    (3, 'Rahul Verma', 'Data Analyst', 'Analytics', 55000),
    (4, 'Sneha Raj', 'Project Manager', 'Management', 85000),
    (5, 'Vikram Singh', 'DevOps Engineer', 'IT', 75000)
])

In [ ]:

cursor.executemany('''
INSERT INTO projects (id, name, status, budget, lead_id)
VALUES (?, ?, ?, ?, ?)
''', [
    (101, 'Employee Management System', 'Completed', 150000, 2),
    (102, 'Cloud Migration', 'In Progress', 300000, 5),
    (103, 'Data Analytics Dashboard', 'In Progress', 200000, 3),
    (104, 'Customer Portal', 'Planning', 250000, 2),
    (105, 'HR Automation', 'Completed', 100000, 4)
])

conn.commit()
conn.close()

### Execute Query ###

In [ ]:
cursor.execute("SELECT * FROM employees")

### Database Content Extraction ###

In [22]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

d:\RAG\RAG-Learning\Repo-1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
# Connect to the SQLite database
db = SQLDatabase.from_uri("sqlite:///data/databases/company.db")

# Get the list of available tables
print(f"Tables: {db.get_usable_table_names()}")

# Get table schema (DDL information)
print("\nTable DDL:")
print(db.get_table_info())

Tables: ['employees', 'projects']

Table DDL:

CREATE TABLE employees (
	id INTEGER, 
	name TEXT NOT NULL, 
	role TEXT, 
	department TEXT, 
	salary REAL, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	role	department	salary
1	Arun Kumar	Software Engineer	IT	60000.0
2	Priya Sharma	Tech Lead	IT	90000.0
3	Rahul Verma	Data Analyst	Analytics	55000.0
*/


CREATE TABLE projects (
	id INTEGER, 
	name TEXT NOT NULL, 
	status TEXT, 
	budget REAL, 
	lead_id INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from projects table:
id	name	status	budget	lead_id
101	Employee Management System	Completed	150000.0	2
102	Cloud Migration	In Progress	300000.0	5
103	Data Analytics Dashboard	In Progress	200000.0	3
*/


In [26]:
from typing import List
from langchain_core.documents import Document

## Method 2 : Custom SQL to Document conversion ##

print("\nCustom SQL Processing")

def process_sql_to_documents(db_path:str) -> List[Document]:
    """Convert SQL database tables to a list of Document objects."""
    # Connect to the SQLite database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    documents = []

    # Get the list of tables in the database
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    for table_name in tables:
        table_name = table_name[0]  # Extract table name from tuple

        # Get the schema of the table
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()
        column_names = [col[1] for col in columns]

        # Get Table Data
        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()

        #Create table overview document
        table_content = f"Table '{table_name}' has columns: {', '.join(column_names)} and {len(rows)} rows."

        # Add Sample records to the document
        table_content += "\nSample Records:\n"
        for row in rows[:5]:  # Limit to first 5 rows for sample
            record = dict(zip(column_names, row))
            table_content+= f"{record}\n"
        doc= Document(
            page_content=table_content,
            metadata={"table": table_name, "type": "overview", "source":db_path}
        )
        documents.append(doc)
        #strategy 2: Create a releationship document for each table
    cursor.execute("""select e.name,e.role,p.name as project_name,
                      p.status from employees 
                      e join projects p on e.id=p.lead_id""")
    relationship_data= cursor.fetchall()
    relationship_doc_content="Employee-Project Relationships:\n"
    for row in relationship_data:
        relationship_doc_content+= f"Employee: {row[0]}, Role: {row[1]}, Project: {row[2]}, Status: {row[3]}\n"
    relationship_doc= Document(
        page_content=relationship_doc_content,
        metadata={"table": "employee_project_relationships", "type": "overview", "source": db_path}
    )
    documents.append(relationship_doc)  
    return documents


Custom SQL Processing


In [27]:
processed_docs = process_sql_to_documents("data/databases/company.db")
print(f"\nProcessed {len(processed_docs)} documents from the SQL database.")
print(processed_docs)


Processed 3 documents from the SQL database.
[Document(metadata={'table': 'employees', 'type': 'overview', 'source': 'data/databases/company.db'}, page_content="Table 'employees' has columns: id, name, role, department, salary and 5 rows.\nSample Records:\n{'id': 1, 'name': 'Arun Kumar', 'role': 'Software Engineer', 'department': 'IT', 'salary': 60000.0}\n{'id': 2, 'name': 'Priya Sharma', 'role': 'Tech Lead', 'department': 'IT', 'salary': 90000.0}\n{'id': 3, 'name': 'Rahul Verma', 'role': 'Data Analyst', 'department': 'Analytics', 'salary': 55000.0}\n{'id': 4, 'name': 'Sneha Raj', 'role': 'Project Manager', 'department': 'Management', 'salary': 85000.0}\n{'id': 5, 'name': 'Vikram Singh', 'role': 'DevOps Engineer', 'department': 'IT', 'salary': 75000.0}\n"), Document(metadata={'table': 'projects', 'type': 'overview', 'source': 'data/databases/company.db'}, page_content="Table 'projects' has columns: id, name, status, budget, lead_id and 5 rows.\nSample Records:\n{'id': 101, 'name': 'Em